# Daily Challenge: How to Finetune LLMs with LoRA

**Parameter-Efficient Fine-Tuning (PEFT)** methods, like **LoRA (Low-Rank Adaptation)**, address the challenges of fine-tuning large language models (LLMs) by only updating a small subset of the model's parameters. This drastically reduces computational and storage costs.

In this challenge we will:
1. Load a pre-trained model (`bigscience/bloomz-560m`) and its tokenizer.
2. Load and preprocess the `Abirate/english_quotes` dataset (10% sample).
3. Configure LoRA with `LoraConfig` and apply it with `get_peft_model`.
4. Train the LoRA-adapted model with the Hugging Face `Trainer`.
5. Save the fine-tuned adapter, reload it with `PeftModel.from_pretrained`, and run inference.

## 1. Install the necessary libraries

We pin `peft==0.4.0` to match the API used in this challenge (`LoraConfig`, `get_peft_model`, `PeftModel`). We also need `datasets` and `transformers`.

In [ ]:
# peft        -> the Parameter-Efficient Fine-Tuning library (provides LoRA).
# datasets    -> Hugging Face library to download/stream the english_quotes dataset.
# transformers-> gives us the model, the tokenizer, the Trainer and TrainingArguments.
# accelerate  -> backend the Trainer relies on to place tensors on CPU/GPU.
# The -q flag keeps the install output quiet; peft is pinned for a stable API.
%pip install -q peft==0.4.0 datasets transformers accelerate

In [ ]:
import os

# Create a local folder to hold intermediate files / model outputs.
# exist_ok=True means the call does NOT raise an error if the folder already exists.
os.makedirs("cache", exist_ok=True)

## 2. Load the pre-trained model and tokenizer

`bigscience/bloomz-560m` is a small (560M parameter) multilingual causal language model — light enough to fine-tune on a CPU when only the LoRA adapters are trained.

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

# The identifier of the pre-trained model on the Hugging Face Hub.
model_name = "bigscience/bloomz-560m"

# The tokenizer turns text into the integer token IDs the model expects (and back).
# AutoTokenizer automatically picks the correct tokenizer class for this model.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# AutoModelForCausalLM loads the model with a language-modeling head, i.e. it predicts
# the next token -> exactly what we need for text generation.
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

# BLOOM ships without a dedicated padding token. When we batch sequences of different
# lengths they must be padded to the same length, so we reuse the end-of-sequence (EOS)
# token as the padding token to avoid an error during collation.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Print the architecture so we can see the layer names (useful for target_modules later).
print(foundation_model)

## 3. Load and preprocess the dataset

We take a **10% sample** of the training split with the slicing syntax `train[:10%]`, then tokenize the `quote` column.

In [ ]:
# Download only the first 10% of the training split.
# The "train[:10%]" slice keeps the download small and the demo fast.
data = load_dataset("Abirate/english_quotes", split="train[:10%]")  # Sample 10%

# Convert each quote string into token IDs.
# .map() applies the function to every row; batched=True feeds many rows at once,
# which is much faster than tokenizing one row at a time.
# The new columns 'input_ids' and 'attention_mask' are added to the dataset.
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# Select only the first 5 examples -> a tiny training set so the CPU demo finishes quickly.
train_sample = data.select(range(5))

# The Trainer's data collator only wants the tokenized columns. The original text columns
# ('quote', 'author', 'tags') cannot be turned into tensors and would crash collation,
# so we drop them and keep just 'input_ids' / 'attention_mask'.
train_sample = train_sample.remove_columns(["quote", "author", "tags"])

# Show the resulting dataset object (features + number of rows).
display(train_sample)

## 4. Configure LoRA with `LoraConfig`

Key parameters:
- **`r`** — the rank of the low-rank update matrices. A small `r` (here `1`) means very few trainable parameters.
- **`lora_alpha`** — a scaling factor for the LoRA weights.
- **`target_modules`** — which layers receive the adapters. For BLOOM the attention projection is named `query_key_value`.
- **`task_type="CAUSAL_LM"`** — autoregressive text generation.

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

# LoraConfig describes HOW the LoRA adapters are built and inserted into the model.
lora_config = LoraConfig(
    r=1,                                  # rank of the low-rank matrices A and B; smaller r = fewer trainable params
    lora_alpha=1,                         # scaling factor applied to the LoRA update (effective scale = alpha / r)
    target_modules=["query_key_value"],   # name of the attention projection in BLOOM where adapters are injected
    lora_dropout=0.05,                    # dropout on the LoRA path to reduce over-fitting on the small dataset
    bias="none",                          # do not train any bias terms (keeps the parameter count minimal)
    task_type="CAUSAL_LM",                # tells PEFT this is a next-token (autoregressive) generation task
)

## 5. Apply LoRA with `get_peft_model`

`get_peft_model` wraps the frozen foundation model and injects the trainable LoRA adapter layers. `print_trainable_parameters()` shows just how few parameters we actually train.

In [ ]:
# Wrap the base model: get_peft_model freezes the original 560M weights and adds the
# small, trainable LoRA adapter layers described by lora_config.
peft_model = get_peft_model(foundation_model, lora_config)

# Print how many parameters are actually trainable vs the total.
# You should see that only a tiny fraction (<<1%) is trainable -> that is the whole point of LoRA.
peft_model.print_trainable_parameters()

## 6. Set up the training arguments

We use a higher learning rate than full fine-tuning (LoRA can tolerate it) and force CPU training for portability.

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer
import os

# Folder where checkpoints and the final adapter will be written.
output_directory = os.path.join("../cache/working", "peft_lab_outputs")

# TrainingArguments collects every knob the Trainer needs.
training_args = TrainingArguments(
    report_to="none",            # disable external loggers (W&B, TensorBoard, etc.)
    output_dir=output_directory, # where checkpoints/logs are saved
    auto_find_batch_size=True,   # let the Trainer automatically pick a batch size that fits in memory
    learning_rate=3e-2,          # higher LR than full fine-tuning; LoRA tolerates large steps
    num_train_epochs=5,          # number of passes over our tiny 5-example training set
    use_cpu=True,                # force CPU so the notebook runs anywhere (no GPU required)
)

## 7. Initialize and train with `Trainer`

`DataCollatorForLanguageModeling(mlm=False)` builds the causal-LM batches (it also creates the `labels` from `input_ids` for us).

In [ ]:
# The Trainer wires together the model, the settings, the data and the collator,
# then handles the whole training loop (forward pass, loss, backprop, optimizer step).
trainer = Trainer(
    model=peft_model,            # the LoRA-wrapped model (only adapters get updated)
    args=training_args,          # the TrainingArguments defined above
    train_dataset=train_sample,  # our tokenized 5-example dataset
    # The data collator pads each batch and, with mlm=False, creates the 'labels' for
    # causal language modeling by copying 'input_ids' (the model learns to predict the next token).
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

# Run the training loop. On CPU this is slow but works for the small demo.
trainer.train()

## 8. Save the fine-tuned LoRA model

`save_pretrained` writes only the small adapter weights (plus the config), not the whole 560M-parameter base model.

In [ ]:
import time

# Use the current Unix timestamp to build a unique folder name so repeated runs
# do not overwrite each other.
time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

# Save ONLY the LoRA adapter weights + config (a few MB), not the full base model.
trainer.model.save_pretrained(peft_model_path)

# Confirm what was written: typically adapter_config.json and adapter_model.bin.
print("Adapter saved to:", peft_model_path)
print(os.listdir(peft_model_path))

## 9. Load the saved LoRA model for inference

We reload a fresh copy of the foundation model and attach the saved adapter with `PeftModel.from_pretrained`. `is_trainable=False` puts it in inference mode.

In [ ]:
from peft import PeftModel

# Step 1: load a clean copy of the original 560M base model.
base_model = AutoModelForCausalLM.from_pretrained(model_name)

# Step 2: layer our trained LoRA adapter on top of that base model.
# is_trainable=False sets the model to inference mode (no further training, no dropout).
loaded_model = PeftModel.from_pretrained(
    base_model,
    peft_model_path,
    is_trainable=False,  # inference only
)

## 10. Generate text with the fine-tuned model

In [ ]:
# Encode the prompt into token IDs. return_tensors="pt" gives PyTorch tensors.
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

# Ask the model to continue the prompt.
outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],            # the encoded prompt
    attention_mask=inputs["attention_mask"],  # tells the model which tokens are real (not padding)
    max_new_tokens=50,                        # generate up to 50 new tokens
    do_sample=True,                           # sample instead of greedy decoding -> more creative text
    top_p=0.9,                                # nucleus sampling: keep the most probable tokens summing to 90%
    temperature=0.8,                          # <1 makes output more focused, >1 more random
    repetition_penalty=1.2,                   # discourage the model from repeating itself
    eos_token_id=tokenizer.eos_token_id,      # stop when the end-of-sequence token is produced
    pad_token_id=tokenizer.pad_token_id,      # padding token id (avoids a warning)
)

# Decode the generated token IDs back into human-readable text.
# skip_special_tokens=True removes tokens like EOS from the output string.
print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

## Summary

We fine-tuned `bloomz-560m` on English quotes using **LoRA**, training only a tiny fraction of the parameters (printed in step 5) while keeping the 560M-parameter base model frozen. We then:
- saved just the lightweight adapter with `save_pretrained`,
- reloaded it onto the base model with `PeftModel.from_pretrained`,
- and generated new quote-style text.

This is the essence of **Parameter-Efficient Fine-Tuning**: adapt a large model to a task at a fraction of the compute and storage cost of full fine-tuning.